# Importa bibliotecas

In [1]:
import pandas as pd
import numpy  as np

# Trata IPTUs

### Cria todos os dicionarios e listas

In [2]:
file_path = "D:\\Drive\\Codigos bee6\\notebooks\\notebooks_bee_brokers\\IPTU_2024\\IPTU_2024.csv"

try:
    # Use 'on_bad_lines' to skip or warn on bad lines
    df = pd.read_csv(file_path, encoding='ISO-8859-1', sep=';', on_bad_lines='skip')
except UnicodeDecodeError as e:
    print("Encoding issue. Trying 'cp1252'.")
    df = pd.read_csv(file_path, encoding='cp1252', sep=';', on_bad_lines='skip')
except pd.errors.ParserError as e:
    print(f"Parser error: {e}")

# Mapeia tudo previamente
map_terreno = {
    "Terr_Normal": ["Normal", "normal"],
    "Terr_Uma_Esquina": ["De esquina", "de esquina"],
    "Terr_Duasoumais_Frentes": ["De duas ou mais frentes", "de duas ou mais frentes não em esquina"],
    "Terr_Interno": ["Terreno interno", "terreno interno"],
    "Terr_Zona_Excl_Resid": ["Lote de esquina em ZER", "de esquina, em ZER(Zona Exclusivamente Residencial)"],
    "Terr_Lote_Fundos": ["Lote de fundos", "lote de fundos"],
    "Terr_Lote_Encravado": ["Lote encravado", "lote encravado"],
    "Terr_Desvio_Ferr": ["desvio ferroviario","de 2 ou mais frentes c/ desvio ferroviario"],
    "Terr_Desconhecido": ['Unknown',',97',',96',',98',',94']}

map_construcao = {
    "Constr_Terreno": ["TERRENO", "TERRENO", "terreno"],
    "Constr_Res_Hor_A": ["Residencial horizontal - padrão A", "Residencial horizontal - padrÃ£o A",'normal','residÃªncia horizontal - PadrÃ£o A'],
    "Constr_Res_Hor_B": ["Residencial horizontal - padrão B", "Residencial horizontal - padrÃ£o B",'residÃªncia horizontal - PadrÃ£o B'],
    "Constr_Res_Hor_C": ["Residencial horizontal - padrão C", "Residencial horizontal - padrÃ£o C",'residÃªncia horizontal - PadrÃ£o C'],
    "Constr_Res_Hor_D": ["Residencial horizontal - padrão D", "Residencial horizontal - padrÃ£o D",'residÃªncia horizontal - PadrÃ£o D'],
    "Constr_Res_Hor_E": ["Residencial horizontal - padrão E", "Residencial horizontal - padrÃ£o E",],
    "Constr_Res_Hor_F": ["Residencial horizontal - padrão F", "Residencial horizontal - padrÃ£o F",'residÃªncia horizontal - PadrÃ£o F'],
    "Constr_Res_Ver_A": ["Residencial vertical - padrão A", "Residencial vertical - padrÃ£o A",'residÃªncia vertical - PadrÃ£o A'],
    "Constr_Res_Ver_B": ["Residencial vertical - padrão B", "Residencial vertical - padrÃ£o B",'residÃªncia vertical - PadrÃ£o B'],
    "Constr_Res_Ver_C": ["Residencial vertical - padrão C", "rResidencial vertical - padrÃ£o C",'Residencial vertical - padrÃ£o C'],
    "Constr_Res_Ver_D": ["Residencial vertical - padrão D", "Residencial vertical - padrÃ£o D",'residÃªncia vertical - PadrÃ£o D'],
    "Constr_Res_Ver_E": ["Residencial vertical - padrão E", "Residencial vertical - padrÃ£o E",'residÃªncia vertical - PadrÃ£o E'],
    "Constr_Res_Ver_F": ["Residencial vertical - padrão F", "Residencial vertical - padrÃ£o F",'residÃªncia vertical - PadrÃ£o F'],
    "Constr_Com_Hor_A": ["Comercial horizontal - padrão A", "Comercial horizontal - padrÃ£o A",'comercial horizontal - PadrÃ£o A'],
    "Constr_Com_Hor_B": ["Comercial horizontal - padrão B", "comercial horizontal - padrÃ£o B",'Comercial horizontal - padrÃ£o B'],
    "Constr_Com_Hor_C": ["Comercial horizontal - padrão C", "Comercial horizontal - padrÃ£o C", "comercial horizontal - Padrão C",'comercial horizontal - PadrÃ£o C'],
    "Constr_Com_Hor_D": ["Comercial horizontal - padrão D", "Comercial horizontal - padrÃ£o D", "comercial horizontal - Padrão D",'comercial horizontal - PadrÃ£o D'],
    "Constr_Com_Hor_E": ["Comercial horizontal - padrão E", "comercial horizontal - Padrão E",'comercial horizontal - PadrÃ£o E','residÃªncia horizontal - PadrÃ£o E','Comercial horizontal - padrÃ£o E'],
    "Constr_Com_Ver_A": ["Comercial vertical - padrão A","comercial vertical - Padrão A",'comercial vertical - PadrÃ£o A','Comercial vertical - padrÃ£o A'],
    "Constr_Com_Ver_B": ["comercial vertical - Padrão B", "Comercial vertical - padrão B",'Comercial vertical - padrÃ£o B','comercial vertical - PadrÃ£o B'],
    "Constr_Com_Ver_C": ["Comercial vertical - padrão C", "comercial vertical - PadrÃ£o C",'Comercial vertical - padrÃ£o C'],
    "Constr_Com_Ver_D": ["Comercial vertical - padrão D", "comercial vertical - Padrão D",'Comercial vertical - padrÃ£o D','comercial vertical - PadrÃ£o D'],
    "Constr_Com_Ver_E": ["Comercial vertical - padrão E", "comercial vertical - Padrão E",'comercial vertical - PadrÃ£o E','Comercial vertical - padrÃ£o E'],
    "Constr_Ed_Gar_A": ["Edifício de garagens - padrão A",'EdifÃ­cio de garagens - padrÃ£o A','edifÃ­cio de garagens - PadrÃ£o A'],
    "Constr_Armazem_Dep_C": ["Oficina/Posto de serviço/Armazém/Depósito/Indústria - padrão C", "Oficina/Posto de serviço/Armazém/Depósito/Indústria - padrÃ£o C"],
    "Constr_Armazem_Dep_D": ["Oficina/Posto de serviço/Armazém/Depósito/Indústria - padrão D",'Oficina/Posto de serviÃ§o/ArmazÃ©m/DepÃ³sito/IndÃºstria - padrÃ£o D '],
    "Constr_Oficina_A": ["Barracão/Telheiro/Oficina - padrão A", "BarracÃ£o/Telheiro/Oficina - padrÃ£o A",'BarracÃ£o/Telheiro/Oficina/Posto de serviÃ§o/ArmazÃ©m/DepÃ³sito/IndÃºstria - PadrÃ£o A'],
    "Constr_Oficina_B": (["Barracão/Telheiro/Oficina/Posto de serviço/Armazém/Depósito/Indústria - padrão B", 
    "BarracÃ£o/Telheiro/Oficina/Posto de serviÃ§o/ArmazÃ©m/DepÃ³sito/IndÃºstria - padrÃ£o B",'BarracÃ£o/Telheiro/Oficina/Posto de serviÃ§o/ArmazÃ©m/DepÃ³sito/IndÃºstria - PadrÃ£o B']),
    "Constr_Oficina_C": (["Barracão/Telheiro/Oficina/Posto de serviço/Armazém/Depósito/Indústria - Padrão C",'Oficina/Posto de serviÃ§o/ArmazÃ©m/DepÃ³sito/IndÃºstria - padrÃ£o C',
    'BarracÃ£o/Telheiro/Oficina/Posto de serviÃ§o/ArmazÃ©m/DepÃ³sito/IndÃºstria - PadrÃ£o C']),
    "Constr_Oficina_D": ["Barracão/Telheiro/Oficina/Posto de serviço/Armazém/Depósito/Indústria - Padrão D",'Oficina/Posto de serviÃ§o/ArmazÃ©m/DepÃ³sito/IndÃºstria - padrÃ£o D','BarracÃ£o/Telheiro/Oficina/Posto de serviÃ§o/ArmazÃ©m/DepÃ³sito/IndÃºstria - PadrÃ£o D'],
    "Constr_Oficina_E": ["Barracão/Telheiro/Oficina/Posto de serviço/Armazém/Depósito/Indústria - Padrão E",'BarracÃ£o/Telheiro/Oficina/Posto de serviÃ§o/ArmazÃ©m/DepÃ³sito/IndÃºstria - PadrÃ£o E'],
    "Constr_Casa_Entret_B": ["Templo/Clube/Ginásio ou Estádio esportivo/Museu/Hipódromo/Cinema/Teatro/Aeroporto/Estações/etc. - padrão B",'Templo/Clube/GinÃ¡sio ou EstÃ¡dio esportivo/Museu/HipÃ³dromo/Cinema/Teatro/Aeroporto/EstaÃ§Ãµes/etc. - padrÃ£o B'],
    "Constr_Casa_Entret_C": ["Templo/Clube/Ginásio ou Estádio esportivo/Museu/Hipódromo/Cinema/Teatro/Aeroporto/Estações/etc. - padrão C",'Templo/Clube/GinÃ¡sio ou EstÃ¡dio esportivo/Museu/HipÃ³dromo/Cinema/Teatro/Aeroporto/EstaÃ§Ãµes/etc. - padrÃ£o C'],
    "Constr_Casa_Entret_D": ["Templo/Clube/Ginásio ou Estádio esportivo/Museu/Hipódromo/Cinema/Teatro/Aeroporto/Estações/etc. - padrão D",'Templo/Clube/GinÃ¡sio ou EstÃ¡dio esportivo/Museu/HipÃ³dromo/Cinema/Teatro/Aeroporto/EstaÃ§Ãµes/etc. - padrÃ£o D'],
    "Constr_Ind_E": ["Industria - Padrao E",'IndÃºstria - padrÃ£o E','Indústria - padrão E','Indústria - padrão E'],
    "Constr_Desconhecida" : ['Unknown']}

map_uso = {
    "Uso_Terr": ["terreno", "Terreno"],
    "Uso_Apto": ["Apartamento em condomínio", "apartamento",'Apartamento em condomÃ\xadnio','PrÃ©dio de apartamento, nÃ£o em condomÃ\xadnio, de uso exclusivamente residencial'],
    "Uso_Res": ["residência", "Residência",'residÃªncia','residÃªncia e outro uso (predominÃ¢ncia residencial)','ResidÃªncia e outro uso (predominÃ¢ncia residencial)','ResidÃªncia'],
    "Uso_Res_Coletiva": (["residência coletiva (mais de uma residência no lote), exclusive cortiço",
    "Residência coletiva, exclusive cortiço (mais de uma residência no lote)", "Residência coletiva, exclusive cortiço (mais de uma residência no lote)",
    'residÃªncia coletiva (mais de uma residÃªncia no lote), exclusive cortiÃ§o','ResidÃªncia coletiva, exclusive cortiÃ§o (mais de uma residÃªncia no lote)']),
    "Uso_Cortico": ["cortiço (habitação coletiva subnormal)", "Cortiço", "Cortiço",'cortiÃ§o (habitaÃ§Ã£o coletiva subnormal)','CortiÃ§o'],
    "Uso_Res_Outros": ["residência e outro uso (predominância residencial)", "Residência e outro uso (predominância residencial)", "Residência e outro uso (predominância residencial)"],
    "Uso_Predio_Res": (["prédio com uso exclusivamente residencial, não em condomínio", 
    "Prédio de apartamento, não em condomínio, de uso misto (apartamentos e escritórios e/ou consultórios), com ou sem loja (predominância residencial)",
    "Prédio de apartamento, não em condomínio, de uso exclusivamente residencial",'prÃ©dio com uso exclusivamente residencial','nÃ£o em condomÃ\xadnio','prÃ©dio com uso exclusivamente residencial, nÃ£o em condomÃ\xadnio']),
    "Uso_Gar_Res": ["garagem em edifício de uso exclusivamente residencial", "Garagem (unidade autônoma) em edifício em condomínio de uso exclusivamente residencial",'garagem em edifÃ\xadcio de uso exclusivamente residencial','Garagem (unidade autÃ´noma) em edifÃ\xadcio em condomÃ\xadnio de uso exclusivamente residencial'],
    "Uso_Gar_Com": (["garagem em edifício de escritórios, consultórios ou misto", 
    "Garagem (unidade autônoma) em edifício em condomínio de escritórios, consultórios ou misto",'garagem em edifÃ\xadcio de escritÃ³rios, consultÃ³rios ou misto','Garagem (unidade autÃ´noma) em edifÃ\xadcio em condomÃ\xadnio de escritÃ³rios, consultÃ³rios ou misto']),
    "Uso_Garagem": ([
    "estacionamento e garagem, não em condomínio","Garagem (exclusive em prédio em condomínio)",'garagem, em prédio de garagens',
    "garagem, em prédio de garagens","Garagem (unidade autônoma) de prédio de garagens",
    "Garagem (exclusive em prédio","Garagem (unidade autônoma) de prédio de garagens, estacionamento e garagem",'garagem, em prÃ©dio de garagens','estacionamento e garagem, nÃ£o em condomÃ\xadnio',
    'Garagem (exclusive em prÃ©dio em condomÃ\xadnio)','Garagem (unidade autÃ´noma) de prÃ©dio de garagens']),
    "Uso_Escr_Consult": (["escritório ou consultório", "Escritório/consultório em condomínio (unidade autônoma)", "Escritório/consultório em condomínio (unidade autônoma)",
    'escritÃ³rio ou consultÃ³rio','PrÃ©dio de escritÃ³rio ou consultÃ³rio, nÃ£o em condomÃ\xadnio, com ou sem lojas','EscritÃ³rio/consultÃ³rio em condomÃ\xadnio (unidade autÃ´noma)']),
    "Uso_Loja": ["loja", "Loja"],
    "Uso_Loja_Res_Com": (["loja e residência (predominância comercial)", "Loja e residência (predominância comercial)", "Loja e residência (predominância comercial)", 
    "Loja em edifício em condomínio (unidade autônoma)",'loja e residÃªncia (predominÃ¢ncia comercial)','loja em edifÃ\xadcio em condomÃ\xadnio',
    'Loja e residÃªncia (predominÃ¢ncia comercial)','Loja em edifÃ\xadcio em condomÃ\xadnio (unidade autÃ´noma)']),
    "Uso_Flat_Res": ["Flat residencial em condomínio", "flat, residencial", "Flat residencial em condomínio",'Flat residencial em condomÃ\xadnio'],
    "Uso_Flat_Com": ["Flat de uso comercial (semelhante a hotel)", "flat, não residencial",'flat, nÃ£o residencial'],
    "Uso_Pred_Escr_Consult": (["prédio de escritório ou consultório, com ou sem lojas, não em condomínio", 
    "Prédio de escritório ou consultório, não em condomínio, com ou sem lojas",
    'prÃ©dio de escritÃ³rio ou consultÃ³rio, com ou sem lojas, nÃ£o em condomÃ\xadnio','prÃ©dio com uso misto, predominÃ¢ncia de uso nÃ£o residencial, nÃ£o em condomÃ\xadnio','PrÃ©dio de escritÃ³rio, nÃ£o em condomÃ\xadnio, de uso misto (apartamentos e escritÃ³rios e/ou consultÃ³rios) com ou sem loja (predominÃ¢ncia comercial)']),
    "Uso_Loja_Edif": ["loja em edifício em condomínio", "Loja em edifício em condomínio (unidade autônoma)"],
    "Uso_Com_Geral": ["outras edificações do tipo (uso comércio), com utilização múltipla", "Outras edificações de uso comercial, com utilização múltipla"],
    "Uso_Industria": ["indústria", "Indústria", "Indústria",'indÃºstria','IndÃºstria'],
    "Uso_Oficina": ["oficina", "Oficina"],
    "Uso_Armaz_Dep": ["armazéns gerais e depósitos", "Armazéns gerais e depósitos", "Armazéns gerais e depósitos",'armazÃ©ns gerais e depÃ³sitos','ArmazÃ©ns gerais e depÃ³sitos'],
    "Uso_Escola": ["escola", "Escola"],
    "Uso_Templo": ["templo", "Templo"],
    "Uso_Posto_Serv": ["posto de serviço (combustíveis)", "Posto de serviço", "Posto de serviço",'posto de serviÃ§o (combustÃ\xadveis)','Posto de serviÃ§o'],
    "Uso_Hotel": ["hotel, pensão ou hospedaria", "Hotel, pensão ou hospedaria", "Hotel, pensão ou hospedaria",'hotel, pensÃ£o ou hospedaria','Hotel, pensÃ£o ou hospedaria'],
    "Uso_Hosp_Saude": (["hospital, ambulatório, casa de saúde e assemelhados", 
    "Hospital, ambulatório, casa de saúde e assemelhados", "Hospital, ambulatório, casa de saúde e assemelhados",
    'hospital, ambulatÃ³rio, casa de saÃºde e assemelhados','Hospital, ambulatÃ³rio, casa de saÃºde e assemelhados']),
    "Uso_Clube_Esp": ["clube esportivo", "Clube esportivo"],
    "Uso_Cine_Teatro": ["cinema, teatro, casa de diversão, clube ou congênere", "Cinema, teatro, casa de diversão, clube ou congênere", "Cinema, teatro, casa de diversão, clube ou congênere",'cinema, teatro, casa de diversÃ£o, clube ou congÃªnere','Cinema, teatro, casa de diversÃ£o, clube ou congÃªnere'],
    "Uso_Radio_Tv": (["estação radioemissora, de televisão ou empresa jornalística", "Estação radioemissora, de televisão ou empresa jornalística",'EstaÃ§Ã£o radioemissora, de televisÃ£o ou empresa jornalÃ\xadstica', 
    "Estaço radioemissora, de televisão ou empresa jornalística",'estaÃ§Ã£o radioemissora, de televisÃ£o ou empresa jornalÃ\xadstica']),
    "Uso_Deposito": ["Depósito Condomínio Residencial"],
    "Uso_Social_Relig": (["asilo, orfanato, creche, seminário ou convento", "Asilo, orfanato, creche, seminário ou convento", 
    "Asilo, orfanato, creche, seminário ou convento",'asilo, orfanato, creche, seminÃ¡rio ou convento','Asilo, orfanato, creche, seminÃ¡rio ou convento']),
    "Uso_Misto": (["prédio com uso misto, predominância de uso não residencial, não em condomínio", 
    "Outras edificações de uso coletivo, com utilização múltipla", "Outras edificações de uso especial, com utilização múltipla",
     "Outras edificações de uso comercial, com utilização múltipla", "Outras edificações de uso de serviço, com utilização múltipla", 
     "outras edificações do tipo (uso serviço), com utilização múltipla", "Outras edificações de uso coletivo, com utilização múltipla",
     "prédio com uso misto, predominância de uso residencial, não em condomínio", "Prédio de apartamento, não em condomínio, de uso exclusivamente residencial", 
     "outras edificações do tipo (uso coletivo), com utilização múltipla", "Outras edificações de uso de serviço, com utilização múltipla", 
     "Prédio de escritório, não em condomínio, de uso misto (apartamentos e escritórios e/ou consultórios) com ou sem loja (predominância comercial)",
     "outras edificações do tipo (uso especial), com utilização múltipla", "Outras edificações de uso especial, com utilização múltipla", "Não residencial",
     'prÃ©dio com uso misto, predominÃ¢ncia de uso residencial, nÃ£o em condomÃ\xadnio','outras edificaÃ§Ãµes do tipo (uso coletivo), com utilizaÃ§Ã£o mÃºltipla',
     'outras edificaÃ§Ãµes do tipo (uso serviÃ§o), com utilizaÃ§Ã£o mÃºltipla','outras edificaÃ§Ãµes do tipo (uso comÃ©rcio), com utilizaÃ§Ã£o mÃºltipla',
     'outras edificaÃ§Ãµes do tipo (uso especial), com utilizaÃ§Ã£o mÃºltipla','PrÃ©dio de apartamento, nÃ£o em condomÃ\xadnio, de uso misto (apartamentos e escritÃ³rios e/ou consultÃ³rios), com ou sem loja (predominÃ¢ncia residencial)',
     'Outras edificaÃ§Ãµes de uso comercial, com utilizaÃ§Ã£o mÃºltipla','Outras edificaÃ§Ãµes de uso especial, com utilizaÃ§Ã£o mÃºltipla','Outras edificaÃ§Ãµes de uso coletivo, com utilizaÃ§Ã£o mÃºltipla',
     'Outras edificaÃ§Ãµes de uso de serviÃ§o, com utilizaÃ§Ã£o mÃºltipla'])}

rename_columns = {
    'ANO DO EXERCICIO': 'Ano',
    'CEP DO IMOVEL': 'Cep',
    'TIPO DE TERRENO': 'Terreno',
    'TIPO DE PADRAO DA CONSTRUCAO': 'Construcao',
    'TIPO DE USO DO IMOVEL': 'Uso',
    'FRACAO IDEAL': 'Fracao_Propriedade',
    'AREA DO TERRENO': 'Area_Terreno',
    'AREA CONSTRUIDA': 'Area_Construida',
    'AREA OCUPADA': 'Area_Ocupada',
    'VALOR DO M2 DO TERRENO': 'm2_Terreno',
    'VALOR DO M2 DE CONSTRUCAO': 'm2_Construcao',
    'QUANTIDADE DE PAVIMENTOS': 'Num_Pavimentos',
    'TESTADA PARA CALCULO': 'Testada',
    'FATOR DE OBSOLESCENCIA': 'Fator_Obsolescencia'}

# List of aggregating columns
agg_columns = ['Ano', 'Udh_Prox', 'Terreno', 'Construcao', 'Uso']

# List of columns to compute the average
avg_columns = [
    'Fracao_Propriedade_Media',
    'Area_Terreno_Media',
    'Area_Construida_Media',
    'Area_Ocupada_Media',
    'm2_Terreno_Medio',
    'm2_Construcao_Medio',
    'Idade_Imovel_Media',
    'Num_Pavimentos_Medio',
    'Testada_Media',
    'Fator_Obsolescencia_Media',
    'Quant_Esquinas_Media']

float_columns = [
    'ANO DE INICIO DA VIDA DO CONTRIBUINTE',
    'ANO DA CONSTRUCAO CORRIGIDO']

# Colunas selecionadas
selected_columns = [
    'NUMERO DO CONTRIBUINTE',
    'CODLOG DO IMOVEL',
    'ANO DE INICIO DA VIDA DO CONTRIBUINTE',
    'ANO DO EXERCICIO', 'CEP DO IMOVEL', 'TIPO DE TERRENO', 'TIPO DE PADRAO DA CONSTRUCAO', 
    'TIPO DE USO DO IMOVEL', 'FRACAO IDEAL', 'AREA DO TERRENO', 'AREA CONSTRUIDA', 'AREA OCUPADA', 
    'VALOR DO M2 DO TERRENO', 'VALOR DO M2 DE CONSTRUCAO', 'ANO DA CONSTRUCAO CORRIGIDO', 
    'QUANTIDADE DE PAVIMENTOS', 'TESTADA PARA CALCULO', 'FATOR DE OBSOLESCENCIA']

C:\Users\guici\AppData\Local\Temp\ipykernel_19428\3593122454.py:5: DtypeWarning: Columns (20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='ISO-8859-1', sep=';', on_bad_lines='skip')


### Tratamento Iptus

In [3]:
# Seleciona as colunas
df = df[['ANO DO EXERCICIO','NUMERO DO CONTRIBUINTE','CODLOG DO IMOVEL','NOME DE LOGRADOURO DO IMOVEL',
        'NUMERO DO IMOVEL','CEP DO IMOVEL','TIPO DE TERRENO','TIPO DE PADRAO DA CONSTRUCAO','TIPO DE USO DO IMOVEL',
        'QUANTIDADE DE ESQUINAS/FRENTES','FRACAO IDEAL','AREA DO TERRENO','AREA CONSTRUIDA','AREA OCUPADA',
        'ANO DA CONSTRUCAO CORRIGIDO','QUANTIDADE DE PAVIMENTOS',
        'TESTADA PARA CALCULO','FATOR DE OBSOLESCENCIA','ANO DE INICIO DA VIDA DO CONTRIBUINTE']] 

columns_to_convert = ['ANO DO EXERCICIO','QUANTIDADE DE ESQUINAS/FRENTES', 'FRACAO IDEAL', 
'AREA DO TERRENO', 'AREA CONSTRUIDA', 'AREA OCUPADA','ANO DA CONSTRUCAO CORRIGIDO','QUANTIDADE DE PAVIMENTOS', 
'TESTADA PARA CALCULO', 'FATOR DE OBSOLESCENCIA', 'ANO DE INICIO DA VIDA DO CONTRIBUINTE']
for col in columns_to_convert:
# Before converting, it's a good idea to replace commas with dots 
# (in case the number format uses commas as decimal separators)
    df[col] = df[col].astype(float)     

# Cria a Idade dos imoveis e do contribuinte
df['Idade_Imovel'] = df['ANO DO EXERCICIO'] - df['ANO DA CONSTRUCAO CORRIGIDO']
df['Idade_Contribuinte'] = df['ANO DO EXERCICIO'] - df['ANO DE INICIO DA VIDA DO CONTRIBUINTE']

# Mapeia terreno, construcao e uso
invert_map = {v_i: k for k, v in map_terreno.items() for v_i in v}
df['TIPO DE TERRENO'] = df['TIPO DE TERRENO'].replace(invert_map)
invert_map = {v_i: k for k, v in map_construcao.items() for v_i in v}
df['TIPO DE PADRAO DA CONSTRUCAO'] = df['TIPO DE PADRAO DA CONSTRUCAO'].replace(invert_map)
invert_map = {v_i: k for k, v in map_uso.items() for v_i in v}
df['TIPO DE USO DO IMOVEL'] = df['TIPO DE USO DO IMOVEL'].replace(invert_map)

# Cria title pra string logradouro
df['NOME DE LOGRADOURO DO IMOVEL'] = df['NOME DE LOGRADOURO DO IMOVEL'].str.title() 

# Deleta oq nao será usado
del df['ANO DE INICIO DA VIDA DO CONTRIBUINTE']
del df['ANO DA CONSTRUCAO CORRIGIDO']

# Renomeia colunas
df = df.rename(columns={
'ANO DO EXERCICIO':'Ano','NUMERO DO CONTRIBUINTE': 'Setor_Quadra_Lote','CODLOG DO IMOVEL': 'Codlog_Imovel',
'NOME DE LOGRADOURO DO IMOVEL': 'Logradouro','NUMERO DO IMOVEL': 'Numero','CEP DO IMOVEL': 'Cep',
'TIPO DE TERRENO' : 'Tipo_Terreno','TIPO DE PADRAO DA CONSTRUCAO': 'Tipo_Construcao',
'TIPO DE USO DO IMOVEL': 'Tipo_Uso','QUANTIDADE DE ESQUINAS/FRENTES':'Quant_Esquinas',
'FRACAO IDEAL': 'Fracao_Uso','AREA DO TERRENO': 'Area_Terreno','AREA CONSTRUIDA': 'Area_Construida',
'AREA OCUPADA' : 'Area_Ocupada','QUANTIDADE DE PAVIMENTOS': 'Pavimentos',
'TESTADA PARA CALCULO': 'Testada','FATOR DE OBSOLESCENCIA': 'Fator_Obsolescencia'})

# Salva o IPTU
df.to_parquet('D:/Drive/Colab Notebooks/Vscode/Duplify/Arquivos/Iptu_Final_2024.parquet')

ValueError: could not convert string to float: 'Residência e outro uso (predominância residencial)'

# Trata os valores Venais

In [44]:
# Seleciona os valores da iptu
ceps = df[['Setor_Quadra_Lote','Cep','Area_Construida']]

# Trata a Setor_Quadra_Lote da venal
venal =  pd.read_csv('D:/Drive/Colab Notebooks/Vscode/Duplify/Arquivos/Valor_Venal.csv')
venal['iptu'] = venal['iptu'].astype(str) 
venal['iptu'] = venal['iptu'].str.zfill(11)
venal['iptu'] = venal['iptu'].str[:10] + '-' + venal['iptu'].str[10:]
venal.columns = ['Setor_Quadra_Lote','Iptu_Anual_Medio','Valor_Venal']

# Mescla e agrupa pela média dos ceps
venal = venal.merge(ceps, how = 'left', on = 'Setor_Quadra_Lote')
venal['Valor_Venal_m2_Medio'] = venal['Valor_Venal'] / venal['Area_Construida']
venal.drop(labels = ['Setor_Quadra_Lote','Valor_Venal','Area_Construida'], axis = 1 , inplace = True)
venal.groupby(by = 'Cep').mean().reset_index()

venal.to_parquet('D:/Drive/Colab Notebooks/Vscode/Duplify/Arquivos/Valor_Venal_Final.parquet')

,Cep,Iptu_Anual_Medio,Valor_Venal_m2_Medio
0,-,6147.007600,inf
1,00275-070,3430.050000,5011.974088
2,01001-000,2494.580000,2311.189625
3,01001-001,8320.070000,2769.752800
4,01001-901,577.928571,2456.053180
...,...,...,...
33471,08490-680,223.930000,1706.212500
33472,08490-685,368.440000,1512.604293
33473,08490-687,288.400000,1337.716667
33474,08490-690,416.720000,1879.285714
